In [1]:
print("Everything is working fine!")

Everything is working fine!


In [2]:
# import dependencies

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22916\3763162838.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
# Load documents and split into chunks
loader = PyPDFLoader(pdf_path)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=75)
docs = text_splitter.split_documents(documents)

In [4]:
print(len(docs))

202


In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    dimensions=768
)

In [6]:
embeddings

OllamaEmbeddings(model='nomic-embed-text', dimensions=768, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [7]:
vdb = Chroma(
    persist_directory="./chroma_db_Vermaanant",
    embedding_function=embeddings
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22916\94224643.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb = Chroma(


In [8]:
vectorstore = vdb.from_documents(
    documents=docs, 
    embedding=embeddings,
    persist_directory="./chroma_db_Vermaanant",
    collection_name="ollama_test")

In [9]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [10]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 5

In [11]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

In [12]:
retriever.invoke("What is the main topic of the paper?")

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'keywords': '', 'source': 'PEFT2312.12148v1.pdf', 'total_pages': 20, 'author': '', 'title': '', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'trapped': '/False', 'creator': 'LaTeX with hyperref', 'page': 19, 'page_label': '20', 'creationdate': '2023-12-20T02:07:04+00:00', 'moddate': '2023-12-20T02:07:04+00:00'}, page_content='Journal.\nFu Lee Wang (Senior Member, IEEE) received\nthe B.Eng. degree in computer engineering and the\nM.Phil. degree in computer science and information\nsystems from The University of Hong Kong, Hong\nKong, and the Ph.D. degree in systems engineering\nand engineering management from The Chinese\nUniversity of Hong Kong, Hong Kong. Prof. Wang\nis the Dean of the School of Science and Tech-\nnology, Hong Kong Metropolitan University, Hong\nKong. He has over 300 publications in international\njournals and conferences and led more than 2

In [13]:
local_llm =ChatOllama(
    model = 'gemma4:e4b',
    max_tokens=512,
    temperature=0.3
)

In [14]:
local_llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, model='gemma4:e4b', temperature=0.3)

In [15]:
prompt_template = """Use the following retrieved documents to answer the question. If you don't know the answer, say I don't know."""


In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(prompt_template)

prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="Use the following retrieved documents to answer the question. If you don't know the answer, say I don't know.")

In [17]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=local_llm,
    chain_type = "stuff",
    retriever=hybrid_retriever,
    verbose=True)

In [18]:
qa_chain

RetrievalQA(verbose=True, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.2', 'langchain': '1.3.14'}}, output_version=None, model='gemma4:e4b', temperature=0.3), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_con

In [20]:
results = qa_chain.invoke("What is the main topic of the paper?")

print(results['result'])



> Entering new RetrievalQA chain...

> Finished chain.
The main topic of the paper is a comprehensive and structured study of **Parameter-Efficient Fine-Tuning (PEFT) methods for PLMs** (Pretrained Language Models).

The paper focuses on:
*   Analyzing and classifying various PEFT techniques.
*   Demonstrating how these methods can be used to fine-tune different types of models (like RoBERTa, T5, and LLaMA) on downstream tasks.
*   Highlighting the benefits of PEFT, such as improving parameter efficiency, lowering memory usage, and reducing the risk of catastrophic forgetting compared to full fine-tuning.


In [21]:
import sys
print(sys.executable)

import langchain
print(langchain.__version__)

C:\Users\Lenovo\Desktop\AI training\RAG\venv\Scripts\python.exe
1.3.14
